In [ ]:
import ee
from IPython.display import Image
import folium
import os

In [ ]:
from utils import (
    get_export_tile_ids,
    get_wgs84_region_bounds,
    get_region_pixel_size,
    get_region_crs,
    get_region_transform,
    gee_grid_tiles,
    get_long_region_name
)

The aim of this script is to:
1. Create the yearly mean 2 m temperature and yearly precipitation sum rasters by overlaying ERA5 (lower resolution) and ERA5 Land (higher resolution) so that ERA5 Land covers the land area and where there is no ERA5 Land cell, ERA5 is used.
2. Create raster tiles covering the region from the overlay.

## 1. Authenticate and initialise GEE

In [ ]:
# provide access key
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "path/to/your/credentials.json"

In [ ]:
ee.Authenticate()

In [ ]:
ee.Initialize(project='your-cloud-project')

## 2. Define the input parameters

In [ ]:
bucket = "bucket_name"
region_name = 'est'
provider = 'era5'

# folder where ERA5 rasters for years 2021-2023 are stored
gee_folder_name = 'projects/your-cloud-project/assets/era5_monthly_rasters_europe'

In [ ]:
# define the years for which to calculate the parameter. Comment out unwanted.

years =  [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

# the parameter names of ERA5 dataset. The parameter name may differ between ERA5 an ERA5 land. Order: ERA5, ERA5 Land.

ERA_parameter_values = {#'mean_temperature': ['mean_2m_air_temperature', 'temperature_2m'],}#\
                       'minimum_temperature': ['minimum_2m_air_temperature', 'temperature_2m_min'],\
                        'maximum_temperature': ['maximum_2m_air_temperature', 'temperature_2m_max'],}#\
                       #'total_precipitation': ['total_precipitation', 'total_precipitation_sum'],
                        #'total_evaporation': ['', 'total_evaporation_sum']}
                        


In [ ]:
# parameter name in the filename

filename_params = {'mean_temperature': 'meantemp',\
                   'minimum_temperature' : 'mintemp',\
                   'maximum_temperature': 'maxtemp',\
                   'total_precipitation': 'totprecip',\
                   'total_evaporation': 'totevap'}
    

## 2. Read in the final grid, get the projection info for tiles

In [ ]:
export_tile_ids = get_export_tile_ids(region_name)

In [ ]:
export_tile_ids

In [ ]:
# uncomment to add tiles for Baltics over the Daugava upstream
# if region_name == 'bal':
#     export_tile_ids.insert(4, '09')
#     export_tile_ids.insert(7, '14')

export_tile_ids

 We will create an output grid where each tile is 10000$\times$10000 pixels, which is the default maximum output image size in GEE. This means that for 30 m resolution the scale parameter will be 300000 (300 km).

In [ ]:
# Region bounds in WGS84
region_bounds = get_wgs84_region_bounds(region_name)

# Pixel size
pixel_size = get_region_pixel_size(region_name)

# Region CRS
region_crs = get_region_crs(region_name)

# Calculate transform
region_transform = get_region_transform(region_name)

# Generate grid tiles
grid_tiles = gee_grid_tiles(region_bounds, region_crs, pixel_size)

In [ ]:
print(region_crs)
    
print(region_transform)

## 3. Specify the tile properties of the final grid

In [ ]:
nodata_val = -9999

In [ ]:
grid_tiles

## 4. Read in ERA5 and ERA5 Land, create the yearly mean rasters

## 4.1. Subselect from GEE assets ERA5 for missing years

In [ ]:
def subselect_and_list_folder(parameter, year):
    """
    filter for the selected parameter and the year in gee assets
    filename convention: era5_monthly_rasters_2020_2023/parameter/year/{region_name}_era5_{parameter}_monthly_{yyyy}-{mm}"""
    
    param = parameter.split("_")
    
    parameter_asset_list = ee.data.listAssets({'parent': gee_folder_name})
    
    # subselect correct parameter
    parameter_foldername = [asset['id'] for asset in parameter_asset_list['assets'] if param[0] in asset['name'] and param[1] in asset['name']]
    
    # returns a list, so need to subselect the 1st element
    year_asset_list = ee.data.listAssets({'parent': parameter_foldername[0]})
    
    # subselect correct year
    year_foldername = [asset['id'] for asset in year_asset_list['assets'] if f"/{year}" in asset['name']]
    
    
    return year_foldername[0]

In [ ]:
# create a collection of ERA5 rasters for years 2021-2023 which are not available in GEE
def create_image_collection_from_assets(parameter, year):
    
    foldername = subselect_and_list_folder(parameter, year)
    
    asset_list = ee.data.listAssets({'parent': foldername})
    
    asset_ids = [asset['id'] for asset in asset_list['assets'] if asset['type'] == 'IMAGE']
    
    print(asset_ids)
    
    # create the image collection from the rasters
    image_collection = ee.ImageCollection([ee.Image(asset_id) for asset_id in asset_ids])
    
    projection = image_collection.first().projection()
    
    return image_collection, projection

## 4.2. Subselect from GEE image collections

In [ ]:
# subset the collection if the year is present in GEE
def subset_collection(collection, parameter):
    
    start_date = f"{year}-01-01"
    end_date = f"{year}-12-31"
    
    image_collection = ee.ImageCollection(collection)
    # The last day is exclusionary, therefore one more day is added.
    subset = image_collection\
        .select(parameter)\
    .filterDate(ee.Date(start_date), ee.Date(end_date).advance(1, "day"))\
        .filterBounds(grid_tiles.geometry())
    
    
    # save the projection info from one unaltered image of the collection (in this case, the first) - needed for resampling
    projection = subset.first().projection()
    
    return subset, projection

In [ ]:
def calculate_yearly_image(collection_subset, parameter):
      
    if 'temperature' in parameter: 
        yearly_image = collection_subset.mean()
        
    elif parameter == 'total_precipitation' or parameter == 'total_evaporation': 
        yearly_image = collection_subset.sum()
        
    
    else:
        print('Unclear how to produce yearly raster (mean/sum). Please update the function calculate_yearly_image')
        
    return yearly_image    

In [ ]:
## functions for visualisation and export of inbetween results
## seperate from final export in case the inbeween results should not
## end up in the bucket.

In [ ]:
def export_image_for_test(image, image_content, projection):
   
    filename = f"{parameter}_{year}_{image_content}"
    
    task = ee.batch.Export.image.toDrive(**{
        'image' : image,
        'description' : filename,
        'folder': 'earthengine',
        'fileNamePrefix': filename,
        'fileFormat': "GeoTIFF",
        'crs': projection.crs().getInfo(),
        'crsTransform': projection.getInfo()["transform"],
        'region': region,
        'maxPixels': 1e9,
    'formatOptions': {
            "noData": nodata_val}});

    task.start() # uncomment to export to drive

In [ ]:
# Visualise the raster by creating a URL to the styled image.

def visualise_resulting_image(image, projection):
    
    # get the min and max values in the image in a dict format
    min_max = image.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=region_name,
    scale=projection.nominalScale().getInfo(),
    bestEffort=True).getInfo()
    
    
    # create the url for image. Selecting the min/max value from the dict
    # by string matching; the dict contains the key '{parameter}_{min/max}'
    url = image.getThumbUrl({
    'min': min_max[[key for key in min_max if key.endswith('min')][0]], \
    'max': min_max[[key for key in min_max if key.endswith('max')][0]],\
        'dimensions': 750,'region': grid_tiles.geometry(),
    'palette': ['black', 'white']})

    # Display the thumbnail 
    display(Image(url=url))


In [ ]:
    ## Function that calls functions of section 4 (from ERA5 reading in and preprocessing until overlay)
    ## and is done on both ERA5 and ERA5 Land the same way

In [ ]:
def prepare_and_visualise_ERA5_raster(collection, parameter_name, collection_name):
    
    # for years which are not present in ERA5 in GEE 
    if (year >= 2020 and collection == "ECMWF/ERA5/MONTHLY") or (parameter == 'total_evaporation' and collection == "ECMWF/ERA5/MONTHLY"): 
        print('select_assets', year)
        str_year = str(year)
        
        subset, projection = create_image_collection_from_assets(parameter, str_year)
    
    # all other instances
    else:
        subset, projection = subset_collection(collection, parameter_name)
    
    
    mean_raster = calculate_yearly_image(subset, parameter)
    
    #export_image_for_test(mean_raster, collection_name, projection)
    
    #visualise_resulting_image(mean_raster, projection)

    
    return mean_raster, projection

## 5. Overlay ERA5 and ERA5 Land

In [ ]:
def overlay_rasters(era5_raster, era5land_raster):

    # 1. resample sea resolution to land resolution (to the desired end resolution) so they can be overlaid
    resampled = era5_raster.reproject(era5land_projection.crs().getInfo(), era5land_projection.getInfo()["transform"])

    # 2. use 'blend' to place land values over the respective sea cells over land areas. 
    overlay = resampled.blend(era5land_raster)
    
    # visualise, check resulting overlay. The image parameter is the same as that of ERA5 (e.g., 2m_air_temperature)
    #visualise_resulting_image(overlay, era5land_projection)
    
    #export_image_for_test(overlay, "overlay", era5land_projection)
    
    return resampled, overlay

## 6. Check if the results are correct

In [ ]:
def test_overlay(overlay, era5land_raster, era5_resampled):
    
    overlay_minus_land = overlay.subtract(era5land_raster) 
    
    # see the unique values. They should not be substantially different than 0.
    unique_values = overlay_minus_land.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=grid_tiles.geometry(),
    scale=era5land_projection.nominalScale().getInfo(),
    maxPixels=1e13 ) 

    #unique_values_dict = unique_values.get(param_ERA5).getInfo()

    #print("Unique values in the Overlay and ERA5 Land raster:", list(unique_values_dict.keys()))
    
    overlay_minus_era = overlay.subtract(era5_resampled)
    
    # plot overlay raster and ERA5 land raster difference. All land cells should be 0, sea -9999
    #visualise_resulting_image(overlay_minus_land, era5land_projection)
    
    # plot overlay raster and ERA5 raster difference. All land cells should be random, sea 0.
    #visualise_resulting_image(overlay_minus_era, era5land_projection)
    
    # export images for test
    #export_image_for_test(overlay_minus_land, "overlay_minus_land", era5land_projection)
    #export_image_for_test(overlay_minus_era, "overlay_minus_era", era5land_projection)
    

## 7.  Get Celsius from Kelvins  / get mm from m 

In [ ]:
def change_units_and_round(overlay, parameter):
    # No NaNs in ERA5, but just in case.. Masking out NaNs so their value doesn't change when converting to Celsius

    unmasked_overlay = overlay.unmask(value=nodata_val) # to overwrite NaNs
    
    masked_overlay = unmasked_overlay.updateMask(overlay.neq(-9999)) # mask so that -9999 is not changed
    
    if 'temperature' in parameter:
        # Celsius degrees = Kelvin degrees - 273.15
        overlay_new_units = masked_overlay.subtract(273.15)
        overlay_rounded = overlay_new_units.multiply(100).round().divide(100)
    
    
    elif parameter == 'total_precipitation' or parameter == 'total_evaporation':
        overlay_new_units = masked_overlay.multiply(1000)
        overlay_rounded = overlay_new_units.round()
        
    else:
        print('The parameter has not been added yet.')
    
    unmasked_new_units = overlay_rounded.unmask(value=nodata_val)
    
    # visualise and export for test
    #visualise_resulting_image(overlay_new_units, era5land_projection)
    
    #export_image_for_test(unmasked_new_units, f"overlay_new_units", era5land_projection)
    
    return unmasked_new_units

## 8. Export the overlaid raster tiles

In [ ]:
def export_final_tiles(final_image):
    
    long_region_name = get_long_region_name(region_name)
    
    # tiles where there is an overlap with the catchment grid.
    tiles_to_export = export_tile_ids
    
    
    # grid ids to list
    grid_ids = grid_tiles.aggregate_array("system:index").getInfo()
    
    for i, grid_id in enumerate(grid_ids):
        
        tile_id = str(i + 1).zfill(2)
        
        if tile_id in tiles_to_export:
            print(f"Tile {tile_id}")
            # Get tile geometry
            feature = ee.Feature(grid_tiles.toList(1, i).get(0))
            region = feature.geometry()

            # name the outputs
            tile_id = str(i + 1).zfill(2)
            filename = "_".join(
                [region_name, provider, filename_param, str(year), tile_id])
            task_name = f"export_{filename}"
            prefix = (f"{long_region_name}/"
                      f"{provider}/"
                      f"{filename_param}/"
                      f"{year}/"
                      f"{filename}")
            

            # export tile
            task = ee.batch.Export.image.toCloudStorage(**{
            'image' : final_image,
            'description' : task_name,
            'bucket': bucket,
            'fileNamePrefix': prefix,
            'fileFormat': "GeoTIFF",
            'crs': region_crs,
            'crsTransform': region_transform,
            'region': region,
            'maxPixels': 1e13,
            'scale': pixel_size,
            'formatOptions': {
                "cloudOptimized": True,
                "noData": nodata_val}});

            task.start()

In [ ]:
for year in years:
    
    for parameter in ERA_parameter_values.keys():
        
        print(year, parameter)
        print("WORKING ON ", year, parameter)
        
        # assign the correct parameters
        param_ERA5 = ERA_parameter_values[parameter][0]
        param_ERA5land = ERA_parameter_values[parameter][1]
        
        
        print(param_ERA5, param_ERA5land)
        # subselect parameter name for the files (e.g., total precipitation - totprecip)
        filename_param = filename_params[parameter]
        
        print(filename_param)
        # select ERA5 and ERA5Land rasters, create the yearly mean image
        era5_mean, era5_projection = prepare_and_visualise_ERA5_raster("ECMWF/ERA5/MONTHLY", param_ERA5, 'era5')
        
        era5land_mean, era5land_projection = prepare_and_visualise_ERA5_raster("ECMWF/ERA5_LAND/MONTHLY_AGGR",\
                                                                               param_ERA5land, 'era5land')
        
        # create the overlay
        resampled_era5, overlay_image = overlay_rasters(era5_mean, era5land_mean)
        
        
        # test the created raster
        test_overlay(overlay_image, era5land_mean, resampled_era5)
        
        # convert to Celsius or to mm
        overlay_to_export = change_units_and_round(overlay_image, parameter)
        
        
        # export the final raster tiles
        export_final_tiles(overlay_to_export)
           